In [2]:
import numpy as np
import json
import pickle

from sklearn.datasets import make_multilabel_classification
from sklearn.neural_network import MLPClassifier as BasePredictor

import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

from scipy.special import softmax

In [3]:
import sys

sys.path.append("..")

In [4]:
from src.data.formatting import resample_data_set
from src.conformal_risk_control.fnr import compute_prediction_set
from src.model_selection.fnr_oracle import compute_oracle_prediction_set
from src.model_selection.fnr_upper import compute_upper_prediction_set
from src.model_selection.fnr_split import select_predictor_index

In [5]:
def reader(path_params):
    with open(path_params, "r") as file:
        params = json.load(file)
    print(params)
    return params

In [6]:
def save_list_with_pickle(my_list, file_name):
    with open(file_name, "wb") as file:
        pickle.dump(my_list, file)

In [7]:
def generate_data_set(params):
    inputs_, outputs_ = make_multilabel_classification(
        n_samples=params["sample_size"] + 1,
        n_features=params["feature_number"],
        n_classes=params["label_number"],
        n_labels=params["avg_label_number"],
        allow_unlabeled=False,
        return_indicator=True,
    )
    return inputs_, outputs_

In [8]:
params_global = reader("params/experiment_4.json")

{'data': {'sample_size': 500, 'feature_number': 300, 'label_number': 10, 'avg_label_number': 4, 'cal_size': 0.3, 'proper_cal_size': 0.5}, 'predictor': {'min_hidden_layer_sizes': [10, 10, 10, 10], 'max_hidden_layer_sizes': [100, 100, 100, 100], 'lam_numbers': [50, 100, 200, 400]}, 'risk': {'control_level': 0.25}}


In [9]:
layer_numbers_ = [
    [
        np.random.randint(
            1, len(params_global["predictor"]["min_hidden_layer_sizes"]) + 1
        )
        for _ in range(lam_number)
    ]
    for lam_number in params_global["predictor"]["lam_numbers"]
]

In [10]:
hidden_layer_sizes__ = [
    [
        tuple(
            [
                np.random.randint(
                    params_global["predictor"]["min_hidden_layer_sizes"][layer_index],
                    params_global["predictor"]["max_hidden_layer_sizes"][layer_index],
                )
                for layer_index in range(layer_number)
            ]
        )
        for layer_number in layer_numbers
    ]
    for layer_numbers in layer_numbers_
]

In [11]:
predictors_ = [
    [
        BasePredictor(
            hidden_layer_sizes=hidden_layer_sizes, max_iter=400, solver="adam"
        )
        for hidden_layer_sizes in hidden_layer_sizes_
    ]
    for hidden_layer_sizes_ in hidden_layer_sizes__
]

In [12]:
rep_number = 100

In [13]:
results_ = [
    {
        "size": {
            "Oracle": [],
            "Upper": [],
            "Split": [],
            "Output": [],
        },
        "FNR": {
            "Oracle": [],
            "Upper": [],
            "Split": [],
        },
    }
    for _ in predictors_
]

In [14]:
results_

[{'size': {'Oracle': [], 'Upper': [], 'Split': [], 'Output': []},
  'FNR': {'Oracle': [], 'Upper': [], 'Split': []}},
 {'size': {'Oracle': [], 'Upper': [], 'Split': [], 'Output': []},
  'FNR': {'Oracle': [], 'Upper': [], 'Split': []}},
 {'size': {'Oracle': [], 'Upper': [], 'Split': [], 'Output': []},
  'FNR': {'Oracle': [], 'Upper': [], 'Split': []}},
 {'size': {'Oracle': [], 'Upper': [], 'Split': [], 'Output': []},
  'FNR': {'Oracle': [], 'Upper': [], 'Split': []}}]

In [ ]:
for predictors, results in zip(predictors_, results_):
    for rep_index in tqdm(range(rep_number)):
        inputs_, outputs_ = generate_data_set(params_global["data"])

        (
            (scaled_inputs_train, outputs_train),
            (scaled_inputs_calibration, outputs_calibration),
            (
                scaled_inputs_selection,
                scaled_inputs_proper_cal,
                outputs_selection,
                outputs_proper_cal,
            ),
            (scaled_input_test, output_test),
        ) = resample_data_set(
            inputs_,
            outputs_,
            params_global["data"]["cal_size"],
            params_global["data"]["proper_cal_size"],
        )

        for predictor in predictors:
            predictor.fit(scaled_inputs_train, outputs_train)

        probas_calibration_per_predictor = [
            softmax(predictor.predict_proba(scaled_inputs_calibration))
            for predictor in predictors
        ]
        proba_test_per_predictor = [
            softmax(predictor.predict_proba(scaled_input_test))
            for predictor in predictors
        ]
        oracle_prediction_set = compute_oracle_prediction_set(
            outputs_calibration,
            probas_calibration_per_predictor,
            output_test,
            proba_test_per_predictor,
            params_global["risk"]["control_level"],
        )

        upper_prediction_set = compute_upper_prediction_set(
            outputs_calibration,
            probas_calibration_per_predictor,
            proba_test_per_predictor,
            params_global["risk"]["control_level"],
        )

        probas_selection_per_predictor = [
            softmax(predictor.predict_proba(scaled_inputs_selection))
            for predictor in predictors
        ]
        split_predictor_index = select_predictor_index(
            outputs_selection,
            probas_selection_per_predictor,
            params_global["risk"]["control_level"],
        )
        split_prediction_set = compute_prediction_set(
            predictors[split_predictor_index],
            scaled_inputs_proper_cal,
            outputs_proper_cal,
            scaled_input_test,
            params_global["risk"]["control_level"],
            True,
        )

        oracle_fnr = (
            np.logical_and(output_test, np.logical_not(oracle_prediction_set)).sum(
                axis=1
            )
            / output_test.sum(axis=1)
        ).item()
        upper_fnr = (
            np.logical_and(output_test, np.logical_not(upper_prediction_set)).sum(
                axis=1
            )
            / output_test.sum(axis=1)
        ).item()
        split_fnr = (
            np.logical_and(output_test, np.logical_not(split_prediction_set)).sum(
                axis=1
            )
            / output_test.sum(axis=1)
        ).item()

        results["FNR"]["Oracle"].append(oracle_fnr)
        results["FNR"]["Upper"].append(upper_fnr)
        results["FNR"]["Split"].append(split_fnr)

        results["size"]["Oracle"].append(oracle_prediction_set.sum())
        results["size"]["Upper"].append(upper_prediction_set.sum())
        results["size"]["Split"].append(split_prediction_set.sum())
        results["size"]["Output"].append(output_test.sum())

        print(
            "Average oracle prediction-set size: {}".format(
                np.mean(results["size"]["Oracle"])
            )
        )
        print("Average output size: {}".format(np.mean(results["size"]["Output"])))
        print(
            "Average split prediction-set size: {}".format(
                np.mean(results["size"]["Split"])
            )
        )
        print(
            "Average upper prediction-set size: {}".format(
                np.mean(results["size"]["Upper"])
            )
        )

        break

  0%|          | 0/100 [00:00<?, ?it/s]c:\Users\razaf\Documents\work\projects\conformal\risk-control\model-selection\code\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\razaf\Documents\work\projects\conformal\risk-control\model-selection\code\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\razaf\Documents\work\projects\conformal\risk-control\model-selection\code\.venv\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (400) reached and the optimization hasn't converged yet.
  warnings.warn(
c:\Users\razaf\Documents\work\projects\conformal\risk-control\model-selec